<a href="https://colab.research.google.com/github/Shreyash0079/DEEP-LEARNING/blob/main/tuner_kreas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [11]:
import pandas as pd
import numpy as np

In [14]:
df=pd.read_csv("/content/diabetes.csv")

In [15]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [16]:
x=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [12]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x=sc.fit_transform(x)

In [23]:
x.shape

(768, 8)

In [21]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)

In [24]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense

In [33]:
model=Sequential()
model.add(Dense(32,activation="relu",input_dim=8))
model.add(Dense(1,activation="sigmoid"))
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.fit(x_train,y_train,batch_size=32,epochs=100,validation_data=(x_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4581 - loss: 16.6475 - val_accuracy: 0.5779 - val_loss: 10.5820
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5100 - loss: 10.3253 - val_accuracy: 0.5909 - val_loss: 6.0530
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4863 - loss: 6.1621 - val_accuracy: 0.5844 - val_loss: 1.9961
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5074 - loss: 1.7825 - val_accuracy: 0.5065 - val_loss: 1.4675
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5747 - loss: 1.3179 - val_accuracy: 0.5195 - val_loss: 1.1192
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5872 - loss: 1.0593 - val_accuracy: 0.5844 - val_loss: 1.0406
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6338 - loss: 0.9855 - val_accuracy: 0.6299 - val_loss: 0.9740
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6447 - loss: 0.8657 - val_accuracy: 0.5584

In [39]:
pip install keras-tuner

In [40]:
import keras_tuner as kt

In [48]:
def build_model(hp):
  model=keras.Sequential()
  model.add(Dense(32,activation="relu",input_dim=8))
  model.add(Dense(1,activation="sigmoid"))

  model.compile(optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop']),
                           loss='binary_crossentropy',
                           metrics=['accuracy'])
  return model

In [49]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [51]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

In [52]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [53]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
def build_model(hp):
  model=keras.Sequential()
  units=hp.Int(8,128 ,step=8)
  model.add(Dense(units,activation="relu",input_dim=8))
  model.add(Dense(1,activation="sigmoid"))
  # units=hp.Int('units',min_value=32,max_value=512,step=32)
  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [56]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [57]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))